>Identifiers

 order_id,
 order_item_id,
 customer_id,
 product_id,

>Time Attributes

order_purchase_timestamp,      order_date (derived)

>Measures

price,
freight_value,
revenue (price + freight_value)

>Dimensional Attributes

customer_state,
product_category_name

>Optional (Good Practice)

payment_value ,
order_status, 
load_date


In [0]:
%sql
USE CATALOG main;

In [0]:
%skip
from pyspark.sql.functions import *
from pyspark.sql.types import *

path="/Volumes/main/ecommerce/lakehouse_vol/silver/"
file_path=dbutils.fs.ls(path)
for i in file_path:
    print(i.name.rstrip("/"))



In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
df_prod=spark.read.table("main.ecommerce.dim_products_scm")
df_cust=spark.read.table("main.ecommerce.dim_customers_scm")

df_ord=spark.read.table("main.ecommerce.sil_orders")
df_items=spark.read.table("main.ecommerce.sil_order_items")
df_pay=spark.read.table("main.ecommerce.sil_payments")

# df_prod.printSchema()
# df_cust.printSchema()



In [0]:
df_fact_sale=df_items.join(df_ord,"order_id","left")\
                   .join(df_cust.filter(col("is_Active_flag")=="Y").select("customer_id","customer_state"),"customer_id","left")\
                   .join(df_prod.filter(col("is_Active_flag")=="Y").select("product_id","product_category_name"),"product_id","left")\
                    .join(df_pay.select("order_id","payment_value"),"order_id","left")\
                   .withColumn("order_date",to_date(col("order_purchase_timestamp"),"yyyy-MM-dd"))\
                   .withColumn("revenue",col("price")+col("freight_value"))\
                   .withColumn("load_date",current_date())

In [0]:
df_fact_sale = df_fact_sale.select(
    col("order_id"),
    col("order_item_id"),
    col("customer_id"),
    col("product_id"),

    col("order_purchase_timestamp"),
    col("order_date"),

    col("price"),
    col("freight_value"),
    col("revenue"),

    col("customer_state"),
    col("product_category_name"),

    col("payment_value"),
    col("order_status"),

    col("load_date")
)
#df_fact_sale.printSchema()
#display(df_fact_sale.limit(5))



In [0]:
df_fact_sale.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("main.ecommerce.fact_sales")
                  #.option("path", "/Volumes/main/ecommerce/lakehouse_vol/silver/fact_sales") #this deside where to store partition  
                  

df_fact_sale.write.format("delta").mode("overwrite").option("overwriteSchema","true").partitionBy("order_date")\
                  .save("/Volumes/main/ecommerce/lakehouse_vol/silver/fact_sale")

In [0]:
%skip
df_fact_sale.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("path", "/mnt/datalake/gold/fact_sales") \
    .partitionBy("order_date") \
    .saveAsTable("main.ecommerce.gold_fact_sales")

In [0]:
%skip
%sql
select * from main.ecommerce.del_customers